In [ ]:
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-i5fy6csi
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-i5fy6csi
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 28f872a2f99a1b201bcd0db14fdbc5a496b9bfd7
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for nvcc4jupyter: filename=nvcc4jupyter-1.2.1-py3-none-any.whl size=10741 sha256=90a829d54142cdbaf5703d6409de30ec0002408286cfbcd108dfdd84cf0ee4fe
  Stored in directory: /tmp/pip-ephem-wheel-cache-bcb_5vus/wheels/7d/b9/66/459b9938664e6a93d1a85323ec52f7e51cd7265d253410a7d8
Successfully built nvcc4jupyter
Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmp3akraoxx".


In [ ]:
cuda_code = r"""
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>
#include <cublas_v2.h>
#include <time.h>
#include <math.h>

// Define matrix indexing for column-major order
#define index(i,j,ld) (((j)*(ld))+(i))

void initializeMatrix(float *matrix, int size) {
    for (int i = 0; i < size; i++) {
        for (int j = 0; j < size; j++) {
            matrix[index(i, j, size)] = (float)(i + j) / size;
        }
    }
}

void cpuMatrixMultiplication(float *A, float *B, float *C, int n) {
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            C[index(i, j, n)] = 0.0f;
            for (int k = 0; k < n; k++) {
                C[index(i, j, n)] += A[index(i, k, n)] * B[index(k, j, n)];
            }
        }
    }
}

int main() {
    int sizes[] = {256, 512, 1024};
    int numSizes = 3;

    for (int s = 0; s < numSizes; s++) {
        int size = sizes[s];
        printf("\nRunning matrix multiplication for size: %d x %d\n", size, size);

        float *A = (float*)aligned_alloc(32, size * size * sizeof(float));
        float *B = (float*)aligned_alloc(32, size * size * sizeof(float));
        float *C_cpu = (float*)aligned_alloc(32, size * size * sizeof(float));
        float *C_gpu = (float*)aligned_alloc(32, size * size * sizeof(float));

        initializeMatrix(A, size);
        initializeMatrix(B, size);

        clock_t start_cpu = clock();
        cpuMatrixMultiplication(A, B, C_cpu, size);
        clock_t end_cpu = clock();

        double time_cpu = ((double)(end_cpu - start_cpu)) / CLOCKS_PER_SEC;
        printf("CPU Matrix Multiplication Time: %f seconds\n", time_cpu);

        // ===== DEVICE MEMORY ALLOCATION =====
        float *d_A, *d_B, *d_C;
        cudaMalloc((void**)&d_A, size * size * sizeof(float));
        cudaMalloc((void**)&d_B, size * size * sizeof(float));
        cudaMalloc((void**)&d_C, size * size * sizeof(float));

        cudaMemcpy(d_A, A, size * size * sizeof(float), cudaMemcpyHostToDevice);
        cudaMemcpy(d_B, B, size * size * sizeof(float), cudaMemcpyHostToDevice);

        // ===== cuBLAS HANDLE =====
        cublasHandle_t handle;
        cublasCreate(&handle);

        float alpha = 1.0f;
        float beta = 0.0f;

        // ===== GPU TIMING =====
        cudaEvent_t start, stop;
        cudaEventCreate(&start);
        cudaEventCreate(&stop);

        cudaEventRecord(start);

        // cuBLAS SGEMM
        cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N,
                    size, size, size,
                    &alpha,
                    d_B, size,
                    d_A, size,
                    &beta,
                    d_C, size);

        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        float time_gpu;
        cudaEventElapsedTime(&time_gpu, start, stop);

        printf("GPU Matrix Multiplication Time (cuBLAS): %f milliseconds\n", time_gpu);

        cudaMemcpy(C_gpu, d_C, size * size * sizeof(float), cudaMemcpyDeviceToHost);

        int errors = 0;
        float max_relative_error = 1e-4;

        for (int i = 0; i < size * size; i++) {
            float relative_error = fabs(C_cpu[i] - C_gpu[i]) /
                                   fmax(fabs(C_cpu[i]), fabs(C_gpu[i]));
            if (relative_error > max_relative_error) {
                errors++;
            }
        }

        if (errors == 0) {
            printf("Results verified successfully for size %d x %d\n", size, size);
        } else {
            printf("Discrepancies found in results\n");
        }

        cublasDestroy(handle);
        cudaFree(d_A);
        cudaFree(d_B);
        cudaFree(d_C);
        free(A);
        free(B);
        free(C_cpu);
        free(C_gpu);
    }

    return 0;
}
"""

with open("matmul.cu", "w") as f:
    f.write(cuda_code)

In [ ]:
!nvcc matmul.cu -o matmul -lcublas

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./matmul


Running matrix multiplication for size: 256 x 256
CPU Matrix Multiplication Time: 0.082031 seconds
GPU Matrix Multiplication Time (cuBLAS): 87.243103 milliseconds
Results verified successfully for size 256 x 256

Running matrix multiplication for size: 512 x 512
CPU Matrix Multiplication Time: 0.837288 seconds
GPU Matrix Multiplication Time (cuBLAS): 0.283456 milliseconds
Results verified successfully for size 512 x 512

Running matrix multiplication for size: 1024 x 1024
CPU Matrix Multiplication Time: 12.295698 seconds
GPU Matrix Multiplication Time (cuBLAS): 0.942848 milliseconds
Results verified successfully for size 1024 x 1024
